In [ ]:
!pip -q install -U transformers accelerate bitsandbytes pymupdf sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 78.4 MB/s eta 0:00:00
   ━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/40.9 MB 146.2 MB/s eta 0:00:01

In [ ]:
import os
import re
import json
import gc
import torch
import pymupdf
import numpy as np

from pathlib import Path
from google.colab import files
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline
)
from sentence_transformers import SentenceTransformer

## LOAD QWEN-3-8B

In [ ]:
from huggingface_hub import login
login()

In [ ]:
MODEL_NAME = "Qwen/Qwen3-8B"
# MODEL_NAME = "Qwen/Qwen3-14B"

print("Loading model...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
    trust_remote_code=True
)

print("Model loaded.")

## UPLOAD PDF

In [ ]:
uploaded = files.upload()

PDF_PATH = list(uploaded.keys())[0]

print("Uploaded:", PDF_PATH)

Saving chapter-03.pdf to chapter-03.pdf
Uploaded: chapter-03.pdf


## Extract text from PDF

In [ ]:
import pymupdf
import re

def extract_pdf(pdf_path):
    doc = pymupdf.open(pdf_path)

    pages = []

    for page_number, page in enumerate(doc, start=1):
        text = page.get_text("text")

        text = text.replace("\x00", " ")
        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r"\n{3,}", "\n\n", text)
        text = text.strip()

        if text:
            pages.append({
                "page": page_number,
                "text": text
            })

    doc.close()

    return pages


pages = extract_pdf(PDF_PATH)

print("Pages extracted:", len(pages))

for page in pages[:2]:
    print(f"\n--- PAGE {page['page']} ---")
    print(page["text"][:1000])

Pages extracted: 43

--- PAGE 1 ---
CHAPTER 3
SOLVING PROBLEMS BY SEARCHING
In which we see how an agent can look ahead to ﬁnd a sequence of actions that will even-
tually achieve its goal.
When the correct action to take is not immediately obvious, an agent may need to plan
ahead: to consider a sequence of actions that form a path to a goal state. Such an agent is
called a problem-solving agent, and the computational process it undertakes is called search.
Problem-solving
agent
Search
Problem-solving agents use atomic representations, as described in Section 2.4.7—that
is, states of the world are considered as wholes, with no internal structure visible to the
problem-solving algorithms. Agents that use factored or structured representations of states
are called planning agents and are discussed in Chapters 7 and 11.
We will cover several search algorithms. In this chapter, we consider only the simplest
environments: episodic, single agent, fully observable, deterministic, static, disc

## Create chunks

In [ ]:
CHUNK_PAGES = 3
CHUNK_OVERLAP_PAGES = 1

def create_chunks(pages):
    chunks = []

    chunk_id = 0

    step = CHUNK_PAGES - CHUNK_OVERLAP_PAGES

    for start in range(
        0,
        len(pages),
        step
    ):

        page_group = pages[
            start:start + CHUNK_PAGES
        ]

        if not page_group:
            break

        combined_text = "\n\n".join(
            page["text"]
            for page in page_group
        )

        chunks.append({
            "chunk_id": chunk_id,
            "pages": [
                page["page"]
                for page in page_group
            ],
            "text": combined_text
        })

        chunk_id += 1

    return chunks

#then run  this code


chunks = create_chunks(pages)

print("Number of chunks:", len(chunks))

for chunk in chunks[:3]:
    print(
        f"\nChunk {chunk['chunk_id']} "
        f"| pages {chunk['pages']}"
    )

    print(chunk["text"][:1000])

Number of chunks: 22

Chunk 0 | pages [1, 2, 3]
CHAPTER 3
SOLVING PROBLEMS BY SEARCHING
In which we see how an agent can look ahead to ﬁnd a sequence of actions that will even-
tually achieve its goal.
When the correct action to take is not immediately obvious, an agent may need to plan
ahead: to consider a sequence of actions that form a path to a goal state. Such an agent is
called a problem-solving agent, and the computational process it undertakes is called search.
Problem-solving
agent
Search
Problem-solving agents use atomic representations, as described in Section 2.4.7—that
is, states of the world are considered as wholes, with no internal structure visible to the
problem-solving algorithms. Agents that use factored or structured representations of states
are called planning agents and are discussed in Chapters 7 and 11.
We will cover several search algorithms. In this chapter, we consider only the simplest
environments: episodic, single agent, fully observable, deterministic, 

## Question-generation prompt

In [ ]:
GENERATION_PROMPT = """
You are an expert university-level educational assessment designer.

Your task is to generate high-quality assessment questions based ONLY on the supplied course material.

The questions will later be used for LLM generation and statistical watermarking. Therefore, questions must be sufficiently detailed and naturally phrased rather than short, one-line questions.

IMPORTANT QUESTION LENGTH REQUIREMENT:

* Every question MUST contain at least 35 words.
* Prefer questions between 35 and 60 words.
* Do NOT make questions longer by adding unnecessary filler or repetition.
* The additional length must come from meaningful content such as context, conditions, relationships between concepts, comparisons, consequences, examples, assumptions, or reasoning requirements.
* Questions should remain clear, natural, grammatically correct, and appropriate for a university-level assessment.
* Avoid artificially repeating the same concept simply to increase length.

DIFFICULTY DEFINITIONS:

EASY:

* Definitions
* Recall
* Terminology
* Basic understanding
* Identifying concepts

Easy questions should still contain sufficient context to reach the required length, but should primarily test recognition, recall, or basic understanding.

MEDIUM:

* Explanation
* Comparison
* Interpretation
* Straightforward application
* Connecting related concepts

Medium questions should require the student to explain relationships, interpret information, compare concepts, or apply a concept in a straightforward situation.

HARD:

* Deeper reasoning
* Analysis
* Multi-step application
* Combining multiple concepts
* Evaluating consequences or trade-offs

Hard questions should require deeper reasoning, analysis, or integration of multiple concepts from the supplied material.

IMPORTANT RULES:

1. Every question must be answerable entirely from the supplied course material.
2. Do not use outside knowledge.
3. Do not provide answers or explanations.
4. Do not create duplicate or near-duplicate questions.
5. Every question must be standalone and understandable without seeing the course material.
6. Every question must be grammatically correct and naturally phrased.
7. Use terminology and concepts that appear in the course material.
8. Each question must focus on a specific concept, relationship, process, or idea from the material.
9. Difficulty MUST be exactly "easy", "medium", or "hard".
10. Avoid yes/no questions.
11. Avoid vague questions such as "Discuss this concept" or "What do you think about this?"
12. Do not mention "the course material", "the text", "the passage", "the document", or "the material" in the question.
13. Do not invent facts, examples, terminology, or scenarios that cannot be supported by the course material.
14. Do not introduce assumptions that require knowledge outside the supplied material.
15. Avoid questions whose answer can be given with only a single word or short phrase.
16. Questions should generally require a multi-sentence explanation or reasoning-based answer.
17. Do not use unnecessary introductory phrases solely to increase question length.
18. Do not combine unrelated concepts merely to make a question longer.
19. Questions must contain at least 35 words and should normally remain below 60 words.
20. Return ONLY a valid JSON array.
21. Do not include markdown.
22. Do not include ```json or any other code fences.
23. Do not include any text before or after the JSON array.

QUESTION QUALITY REQUIREMENTS:

Before producing each question, internally verify that:

* The question is at least 35 words long.
* The question is answerable from the supplied material.
* The question tests the specified difficulty level.
* The question focuses on a meaningful concept.
* The question does not require outside knowledge.
* The question is not a duplicate of another generated question.
* The question cannot be answered adequately with a one-word or very short response.
* The question contains meaningful information rather than filler.
* The wording is natural and appropriate for a university examination.

LENGTH AND QUALITY ARE BOTH REQUIRED:
Do not sacrifice question quality merely to satisfy the word-count requirement. If a concept cannot naturally support a 35-word question, construct a meaningful question around its definition, characteristics, relationships, applications, consequences, or comparison with another concept explicitly discussed in the supplied material.

Generate exactly:

3 easy questions
4 medium questions
3 hard questions

Return exactly this JSON structure:

[
{{
"topic": "...",
"difficulty": "easy",
"question": "..."
}}
]

COURSE MATERIAL:

{TEXT}
"""


## Function to run Qwen

In [ ]:
def generate_with_qwen(prompt, max_new_tokens=1800):

    messages = [
        {
            "role": "system",
            "content": (
                "You are a precise educational dataset generator. "
                "Follow the requested JSON format exactly."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.6,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    result = tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

    return result.strip()

## Parse the JSON returned by Qwen

In [ ]:
def extract_json_array(text):

    text = text.strip()

    # Remove markdown fences
    text = re.sub(r"```json", "", text, flags=re.IGNORECASE)
    text = re.sub(r"```", "", text)

    text = text.strip()

    # Find first [ and last ]
    start = text.find("[")
    end = text.rfind("]")

    if start == -1 or end == -1:
        return None

    json_text = text[start:end + 1]

    try:
        return json.loads(json_text)

    except json.JSONDecodeError:

        # Try to repair common formatting problems
        json_text = json_text.replace(
            "\n",
            " "
        )

        try:
            return json.loads(json_text)
        except:
            return None

## Generate questions from Chapter X

In [ ]:
all_questions = []

for i, chunk in enumerate(chunks):

    print(
        f"\n{'=' * 60}\n"
        f"Processing chunk {i + 1}/{len(chunks)} "
        f"| pages {chunk['pages']}\n"
        f"{'=' * 60}"
    )

    prompt = GENERATION_PROMPT.format(
        TEXT=chunk["text"]
    )

    raw_output = generate_with_qwen(
        prompt,
        max_new_tokens=1800
    )

    questions = extract_json_array(raw_output)

    if questions is None:

        print("Could not parse JSON.")
        print(raw_output[:2000])
        continue

    print("Generated:", len(questions))

    for q in questions:

        if not isinstance(q, dict):
            continue

        topic = str(
            q.get("topic", "")
        ).strip()

        difficulty = str(
            q.get("difficulty", "")
        ).strip().lower()

        question = str(
            q.get("question", "")
        ).strip()

        if (
            not topic
            or not question
            or difficulty not in {
                "easy",
                "medium",
                "hard"
            }
        ):
            continue

        all_questions.append({
            "topic": topic,
            "difficulty": difficulty,
            "question": question,

            # Keep metadata temporarily.
            "_chunk_id": chunk["chunk_id"],
            "_source_pages": chunk["pages"]
        })

    # Free GPU cache
    gc.collect()
    torch.cuda.empty_cache()


print(
    "\nTotal generated questions:",
    len(all_questions)
)


Processing chunk 1/22 | pages [1, 2, 3]
Generated: 9

Processing chunk 2/22 | pages [3, 4, 5]
Generated: 9

Processing chunk 3/22 | pages [5, 6, 7]
Generated: 9

Processing chunk 4/22 | pages [7, 8, 9]
Generated: 9

Processing chunk 5/22 | pages [9, 10, 11]
Generated: 9

Processing chunk 6/22 | pages [11, 12, 13]
Generated: 10

Processing chunk 7/22 | pages [13, 14, 15]


KeyboardInterrupt: 

## Inspect generated questions

In [ ]:
for i, q in enumerate(all_questions[:20], start=1):

    print(f"\n{i}.")
    print("Topic:", q["topic"])
    print("Difficulty:", q["difficulty"])
    print("Question:", q["question"])


1.
Topic: Intelligent Agents
Difficulty: easy
Question: Explain how an agent interacts with its environment through sensors and actuators, and provide an example of how a human agent perceives and acts upon its surroundings using these components.

2.
Topic: Intelligent Agents
Difficulty: easy
Question: Define the term 'percept sequence' and describe why it is crucial for an agent's decision-making process, emphasizing its role in capturing the agent's historical interactions with the environment.

3.
Topic: Intelligent Agents
Difficulty: easy
Question: What is the primary distinction between an agent function and an agent program, and how does this distinction influence the design and implementation of artificial agents according to the material?

4.
Topic: Intelligent Agents
Difficulty: medium
Question: Compare the vacuum-cleaner agent's behavior in a fully observable versus a partially observable environment, and explain how the properties of the environment directly impact the com

## VALIDATION

Now we use the same local Qwen model as a quality-control model.

In [ ]:
VALIDATION_PROMPT = """
You are a strict educational dataset quality evaluator.

Evaluate the generated questions against the supplied course material.

For each question determine:

- Is it a valid question?
- Is it answerable from the source?
- Is it grounded entirely in the source?
- Is the topic correct?
- Is the difficulty correct?
- Is it clear?
- Does it require outside knowledge?
- Is it a duplicate or near duplicate?

Difficulty:

EASY:
Recall, definitions, terminology, basic understanding.

MEDIUM:
Explanation, comparison, interpretation, straightforward application.

HARD:
Reasoning, analysis, multi-step application, or combining concepts.

A question is valid ONLY if it is well grounded in the course material.

Return ONLY this JSON:

[
{{
"id": 0,
"valid": true,
"grounded": true,
"topic_correct": true,
"difficulty_correct": true,
"clear": true
}}
]

COURSE MATERIAL:

{TEXT}

QUESTIONS:

{QUESTIONS}
"""


## Validation function

In [ ]:
def validate_questions_batch(questions, source_text):

    clean_questions = []

    for i, q in enumerate(questions):

        clean_questions.append({
            "id": i,
            "topic": q["topic"],
            "difficulty": q["difficulty"],
            "question": q["question"]
        })

    prompt = VALIDATION_PROMPT.format(
        TEXT=source_text,
        QUESTIONS=json.dumps(
            clean_questions,
            ensure_ascii=False,
            indent=2
        )
    )

    raw_output = generate_with_qwen(
        prompt,
        max_new_tokens=1200
    )

    evaluations = extract_json_array(
        raw_output
    )

    if evaluations is None:

        print("Validation JSON failed.")
        print(raw_output[:2000])

        return []

    valid = []

    for q, evaluation in zip(
        questions,
        evaluations
    ):

        if (
            evaluation.get("valid") is True
            and evaluation.get("grounded") is True
            and evaluation.get("topic_correct") is True
            and evaluation.get("difficulty_correct") is True
            and evaluation.get("clear") is True
        ):

            valid.append(q)

    return valid

## Run validation

In [ ]:
validated_questions = []

# Group questions according to their source chunk.
questions_by_chunk = {}

for q in all_questions:

    chunk_id = q["_chunk_id"]

    if chunk_id not in questions_by_chunk:
        questions_by_chunk[chunk_id] = []

    questions_by_chunk[chunk_id].append(q)


chunk_lookup = {
    c["chunk_id"]: c
    for c in chunks
}


for chunk_id, questions in questions_by_chunk.items():

    print(
        f"\nValidating chunk {chunk_id}: "
        f"{len(questions)} questions"
    )

    # Validate in batches
    batch_size = 5

    for start in range(
        0,
        len(questions),
        batch_size
    ):

        batch = questions[
            start:start + batch_size
        ]

        valid = validate_questions_batch(
            batch,
            chunk_lookup[chunk_id]["text"]
        )

        validated_questions.extend(valid)

        print(
            f"Accepted {len(valid)}/{len(batch)}"
        )


print(
    "\nTotal after validation:",
    len(validated_questions)
)


Validating chunk 0: 10 questions
Accepted 5/5
Accepted 5/5

Validating chunk 1: 9 questions
Accepted 5/5
Accepted 4/4

Validating chunk 2: 9 questions
Accepted 5/5
Accepted 4/4

Validating chunk 3: 9 questions
Accepted 5/5
Accepted 4/4

Validating chunk 4: 9 questions
Accepted 5/5
Accepted 4/4

Validating chunk 5: 9 questions
Accepted 5/5
Accepted 4/4

Validating chunk 6: 9 questions
Accepted 5/5
Accepted 4/4

Validating chunk 7: 9 questions
Accepted 5/5
Accepted 4/4

Validating chunk 8: 9 questions
Accepted 5/5
Accepted 4/4

Validating chunk 9: 9 questions
Accepted 5/5
Accepted 4/4

Validating chunk 10: 9 questions
Accepted 5/5
Accepted 4/4

Validating chunk 11: 9 questions
Accepted 5/5
Accepted 4/4

Validating chunk 12: 9 questions
Accepted 5/5
Accepted 4/4

Validating chunk 13: 9 questions
Accepted 5/5
Accepted 2/4

Total after validation: 125


## Exact duplicate removal

In [ ]:
def normalize_for_duplicate(text):

    text = text.lower()

    text = re.sub(
        r"[^\w\s]",
        "",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


unique_questions = []
seen = set()

for q in validated_questions:

    normalized = normalize_for_duplicate(
        q["question"]
    )

    if normalized in seen:
        continue

    seen.add(normalized)

    unique_questions.append(q)


print(
    "After exact deduplication:",
    len(unique_questions)
)

After exact deduplication: 125


## Semantic deduplication

In [ ]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

questions_text = [
    q["question"]
    for q in unique_questions
]

embeddings = embedding_model.encode(
    questions_text,
    normalize_embeddings=True,
    show_progress_bar=True
)

embeddings = np.array(embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

## REMOVE SIMILAR QUESTIONS

In [ ]:
SIMILARITY_THRESHOLD = 0.88

keep_indices = []

for i in range(
    len(unique_questions)
):

    duplicate = False

    for j in keep_indices:

        similarity = np.dot(
            embeddings[i],
            embeddings[j]
        )

        if similarity >= SIMILARITY_THRESHOLD:

            duplicate = True
            break

    if not duplicate:
        keep_indices.append(i)


final_questions = [
    unique_questions[i]
    for i in keep_indices
]


print(
    "Before semantic deduplication:",
    len(unique_questions)
)

print(
    "After semantic deduplication:",
    len(final_questions)
)

Before semantic deduplication: 125
After semantic deduplication: 123


## Save the final dataset

In [ ]:
final_dataset = []

for q in final_questions:

    final_dataset.append({
        "topic": q["topic"],
        "difficulty": q["difficulty"],
        "question": q["question"]
    })

chapter_name = Path(PDF_PATH).stem

OUTPUT_FILE = f"{chapter_name}.jsonl"

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    for q in final_dataset:

        f.write(
            json.dumps(
                q,
                ensure_ascii=False
            )
            + "\n"
        )


print(
    f"Saved {len(final_dataset)} questions to {OUTPUT_FILE}"
)

Saved 123 questions to chapter-02.jsonl


## Check your distribution

In [ ]:
from collections import Counter

difficulty_counts = Counter(
    q["difficulty"]
    for q in final_dataset
)

topic_counts = Counter(
    q["topic"]
    for q in final_dataset
)

print("\nDIFFICULTY DISTRIBUTION")

for difficulty in [
    "easy",
    "medium",
    "hard"
]:

    print(
        difficulty,
        ":",
        difficulty_counts[difficulty]
    )


print("\nTOPICS")

for topic, count in topic_counts.most_common(30):

    print(
        f"{topic}: {count}"
    )


DIFFICULTY DISTRIBUTION
easy : 41
medium : 42
hard : 40

TOPICS
Intelligent Agents: 74
Performance Measures: 3
Task Environments: 2
Model-Based Reflex Agents: 2
Rationality: 1
Agent Design: 1
Rational vs. Omniscient Agents: 1
Performance Measure Design: 1
Agent Adaptation: 1
Rationality and Uncertainty: 1
Rational Agents: 1
Rationality vs. Omniscience: 1
PEAS Framework: 1
Rational Agent Behavior: 1
Autonomy in Agents: 1
Task Environment Design: 1
Performance Trade-offs: 1
Learning and Adaptation: 1
Environment Complexity: 1
Observability: 1
Multiagent Systems: 1
Sensor and Actuator Design: 1
Environment Properties: 1
Agent Interaction: 1
Task Environment Complexity: 1
Goal-Based Agents: 1
Utility Functions: 1
Model vs Goal-Based Agents: 1
Transition and Sensor Models: 1
Utility-Based Decision Making: 1


## DOWNLOAD

In [ ]:
files.download(OUTPUT_FILE)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>